#### Import

In [1]:
import ipywidgets as widgets
from ipywidgets import interact, interact_manual, IntSlider
from IPython.display import display, clear_output
import re
import os
import os.path

from typing import Tuple, Optional, List, Dict

Comment = ("#", "!", ";", "//")
NUM_RE = re.compile(r"[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?")

#### Functions

In [2]:
def strip_comments(line):
    com = line.strip()
    for p in Comment:
        if com.startswith(p):
            return ""
    return com

def find_num(text):
    """Extract first numeric token from a string like '2500 K' or '0.1 micron'."""
    num = NUM_RE.search(text)
    if not num:
        raise ValueError(f"Expected a number, got: {text!r}")
    return float(num.group(0))


def find_list(text):
    """Extract all numeric tokens from a string like '10000, 2500 K'."""
    nums = [float(x) for x in NUM_RE.findall(text)]
    if not nums:
        raise ValueError(f"Expected one or more numbers, got: {text!r}")
    return nums


def less_value(val, cond):
    val = find_num(val.strip())  # Remove spaces and newline characters
    if not val<cond:  # Check condition
        raise ValueError("File Incorrectly formatted, check documentation")
    return val
#    else raise ValueError("Missing numerical input")
#take whole string return if it is comment or not comment or =, then what is after 
#Make list of numbers/ inputs, check difference of list to reference lists--> run error

def line_check(line):
    clean = strip_comments(line)
    if not clean:
        return None, None
    if '=' in clean:
        # Split only on the first '=' to avoid breaking values with '=' inside
        key, value = line.split('=', 1)
        return key.strip(), value.strip()
    return None, clean

def list_check(lst):
    splits = [x for x in lst.split(',')]
    check = []
    for split in splits: 
        nums = find_list(split)
        if len(nums) != 1:
            raise ValueError("Attach only one number per list item")
        check.append(nums[0])
    return check

def increasing(list):
    for i in range(1, len(list)):
        if list[i] != 0: 
            if not (list[i]>list[i-1]):
                raise ValueError("Values must be increasing")

#### Dat File Functions

In [3]:
def stellar_spectrum_dat_check(file_name):
    """
    Function checks the stellar spectrum .dat file for the following:
        General:
            - Checks if the file exists
            - Checks if the file is in a .dat format
        Specific to stellar spectrum:
            - File has a three-line header and at least one data line
            - Two columns of data
            - Data has ≤ 10000 points
            - Wavelength data column is ascending or descending in values     
    """

    file_name = file_name.strip()
    
    #Checks if file is there
    try:
        exists = os.path.isfile(file_name)
        if not exists:
            raise FileNotFoundError (f'File not found: {file_name}')
    except Exception as err:
        print(err)
        return False
        
    else:
        #Checks if file is a .dat file
        if not file_name.lower().endswith('.dat'):
            raise TypeError("File is not a .dat file")

        #Opens .dat file
        try: 
            with open(file_name, 'r') as file:
                lines = file.readlines()
        except Exception as err:
            print(err)
            return False
                
        #Checks if there are at least 4 lines
        if len(lines) < 4:
            raise ValueError ("File has fewer than 4 lines. A three-line header and at least one row of data are required.")

        header = lines[:3]
        data = [ln.strip() for ln in lines[3:] if ln.strip() != ""]      
            #ignores empty lines

        wavelengths = [] 
            
        for line in data:
            #Checks for two column format
            columns = line.split()
            if len(columns) != 2:
                raise ValueError ("Data must be in a two column format.")

            #Checks to see if data is all numbers
            try:
                wavelength = float(columns[0])
                sed = float(columns[1])
                wavelengths.append(wavelength)
            except ValueError as err:
                print(err)

        #Checks to see if it's over 10000 data points
        if len(data) > 10000:
            raise ValueError ("There must be ≤ 10000 data points.")

        #Checks if wavelengths are ascending or descending
        if len(wavelengths) >= 2:
            if wavelengths != sorted(wavelengths) and wavelengths != sorted(wavelengths, reverse = True):
                raise ValueError ("Wavelengths must be in ascending or descending order.")


def chemical_comp_dat_check(file_name):
    """
    Function checks the chemical composition .dat file for the following:
        General:
            - Checks if the file exists
            - Checks if the file is in a .dat format
        Specific to chemical composition:
            - File has a three-line header and at least one data line
            - Three columns of data 
            - Data has ≤ 10000 points
            - Wavelength data column is ascending or descending in values     
    """

    file_name = file_name.strip()
    
    #Checks if file is there
    try:
        exists = os.path.isfile(file_name)
        if not exists:
            raise FileNotFoundError (f'File not found: {file_name}')
    except Exception as err:
        print(err)
        return False
        
    else:
        #Checks if file is a .dat file
        if not file_name.lower().endswith('.dat'):
            raise TypeError("File is not a .dat file")

        #Opens .dat file
        try: 
            with open(file_name, 'r') as file:
                lines = file.readlines()
        except Exception as err:
            print(err)
            return False
                
        #Checks if there are at least 4 lines
        if len(lines) < 4:
            raise ValueError ("File has fewer than 4 lines. A three-line header and at least one row of data are required.")

        header = lines[:3]
        data = [ln.strip() for ln in lines[3:] if ln.strip() != ""]      
            #ignores empty lines

        wavelengths = [] 
            
        for line in data:
            #Checks for three column format
            columns = line.split()
            if len(columns) != 3:
                raise ValueError ("Data must be in a three column format.")

            #Checks to see if data is all numbers
            try:
                wavelength = float(columns[0])
                absorption_cs = float(columns[1])
                scattering_cs = float(columns[2])
                wavelengths.append(wavelength)
            except ValueError as err:
                print(err)

        #Checks to see if it's over 10000 data points
        if len(data) > 10000:
            raise ValueError ("There must be ≤ 10000 data points.")

        #Checks if wavelengths are ascending or descending
        if len(wavelengths) >= 2:
            if wavelengths != sorted(wavelengths) and wavelengths != sorted(wavelengths, reverse = True):
                raise ValueError ("Wavelengths must be in ascending or descending order.")


def density_dat_check(file_name):
    """
    Function checks the density profile .dat file for the following:
        General:
            - Checks if the file exists
            - Checks if the file is in a .dat format
        Specific to density profile:
            - File has a three-line header and at least one data line
            - Two columns of data 
            - Data has ≤ 1000 points
            - Radius data column is ascending or descending in values     
    """
    
    #Checks if file is there

    file_name = file_name.strip()
    
    try:
        exists = os.path.isfile(file_name)
        if not exists:
            raise FileNotFoundError (f'File not found: {file_name}')
    except Exception as err:
        print(err)
        return False
        
    else:
        #Checks if file is a .dat file
        if not file_name.lower().endswith('.dat'):
            raise TypeError("File is not a .dat file")

        #Opens .dat file
        try: 
            with open(file_name, 'r') as file:
                lines = file.readlines()
        except Exception as err:
            print(err)
            return False
                
        #Checks if there are at least 4 lines
        if len(lines) < 4:
            raise ValueError ("File has fewer than 4 lines. A three-line header and at least one row of data are required.")

        header = lines[:3]
        data = [ln.strip() for ln in lines[3:] if ln.strip() != ""]      
            #ignores empty lines

        radii = [] 
            
        for line in data:
            #Checks for two column format
            columns = line.split()
            if len(columns) != 2:
                raise ValueError ("Data must be in a two column format.")

            #Checks to see if data is all numbers
            try:
                radius = float(columns[0])
                density = float(columns[1])
                radii.append(radius)
            except ValueError as err:
                print(err)

        #Checks to see if it's over 10000 data points
        if len(data) > 1000:
            raise ValueError ("There must be ≤ 1000 data points.")

        #Checks if radii are ascending or descending
        if len(radii) >= 2:
            if radii != sorted(radii) and radii != sorted(radii, reverse = True):
                raise ValueError ("Radii must be in ascending or descending order.")


def optical_dat_check(file_name):
    """
    Function checks the optical depth .dat file for the following:
        General:
            - Checks if the file exists
            - Checks if the file is in a .dat format
        Specific to optical depth:
            - File has an arbitrary number of text lines as the header
                - This text must not contain the equal sign
            - A fiducial wavelength is provided (which is preceded by the equal sign)
            - One column of data
                - Does not contain the equal sign
            - Data has ≤ 999 points
    """
    file_name = file_name.strip()
    
    #Checks if file is there
    try:
        exists = os.path.isfile(file_name)
        if not exists:
            raise FileNotFoundError (f'File not found: {file_name}')
    except Exception as err:
        print(err)
        return False
        
    else:
        #Checks if file is a .dat file
        if not file_name.lower().endswith('.dat'):
            raise TypeError("File is not a .dat file")

        #Opens .dat file
        try: 
            with open(file_name, 'r') as file:
                lines = file.readlines()
        except Exception as err:
            print(err)
            return False
    
        header = []
        data_index = None

        #Separates header and data based on equal sign
        for idx, ln in enumerate(lines):
            if '=' in ln:
                data_index = idx
                break
            header.append(ln)

        data = [ln.strip() for ln in lines[data_index + 1:] if ln.strip() != ""]
        
        #Checks for fiducial wavelength
        if data_index is None:
            raise ValueError ("Missing fiducial wavelength. Header must be followed by a line containing '=' which is the fiducial wavelength.")
            
        #Checks there's no equal sign in header  
        for equal in header:
            if '=' in equal:
                raise ValueError("Header lines must not contain '='.")

        #Checks there is at least 1 data point
        if len(data) < 1:
            raise ValueError ("Data has fewer than 1 line. At least 1 line is required after the fiducial wavelength.")

        for line in data:
            #Checks there's no equal sign in the data
            if '=' in line:
                raise ValueError("Data must not contain '='.")
                
            #Checks for one column format
            columns = line.split()
            if len(columns) != 1:
                raise ValueError ("Data must be in a one column format.")

            #Checks to see if data is all numbers
            try:
                depth = float(columns[0])
            except ValueError as err:
                print(err)

        #Checks to see if it's over 999 data points
        if len(data) > 999:
            raise ValueError ("There must be ≤ 999 data points.")

#### Check function

In [4]:
def check_inp(inp):
    with open(inp, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    keyvalue={}
    cleaned=[]
    key_line={}
    for line, string in enumerate(lines, start=1):
        key, value = line_check(string)
        if key is None and value is None:
            continue
        cleaned.append(strip_comments(string).strip())
        if key is not None:
            key = key.strip().lower()
            if key not in keyvalue:
                keyvalue[key] = value
                key_line[key] = line

                
    #Physical Paramaters
    if "spectrum" not in keyvalue: 
        raise ValueError("Missing spectrum information or heading")
    spectrum = find_num(keyvalue["spectrum"])
    if int(spectrum) not in (1, 2, 3, 4, 5, 6):
        raise ValueError("Spectrum type must be an integer up to value of 6")
    spectrum = int(spectrum)

    if spectrum == 1:
        if "number_of_bb" in keyvalue:
            if int(find_num(keyvalue["number_of_bb"]))<11:
                BB = int(find_num(keyvalue["number_of_bb"]))
            else:
                raise ValueError("(1) Maximum number of BB is 10")
        else:
            raise ValueError("(1) Require Number of BB or flag")

        temps = keyvalue.get("temperatures")
        if temps is None:
            raise ValueError("(1) Missing Temperatures")
        temps = list_check(temps)
        if len(temps) != BB:
            raise ValueError("(1) Need a Temperature value for each Black Body entered")
        if any (temp<=0 for temp in temps) or any (temp>=1e10 for temp in temps):
            raise ValueError("(1) Invalid Temperature")

        if int(find_num(keyvalue["number_of_bb"]))>1:
            lums = keyvalue.get("luminosities")
            if lums is None:
                raise ValueError("(1) Missing Luminosities")
            lums = list_check(lums)
            if BB != 1:
                if len(lums) != BB:
                    raise ValueError("(1) Need a Luminosity value for each Black Body entered")
                if any (lum<=0 for lum in lums):
                    raise ValueError("(1) Invalid Luminosity")

    elif spectrum == 2:
        if "tbb" not in keyvalue:
            raise ValueError("(2) Missing Tbb")
        Tbb = find_num(keyvalue["tbb"])
        if Tbb<0:
            raise ValueError("(2) Tbb must be greater than 0K")

        if "sio_fd" not in keyvalue:
            raise ValueError("(2) Missing SiO feature depth") 
        SiO = find_num(keyvalue["sio_fd"])
        if not (0<= SiO <=100):
            raise ValueError("(2) SiO feature depth must be between 0 and 100%")


    elif spectrum == 3:
        if "pl_n" not in keyvalue:
            raise ValueError("(3) Missing Number of power law segments")
        n = int(find_num(keyvalue["pl_n"]))
        if n<1:
            raise ValueError("(3) Number of power law segments must be at least 1")

        if "pl_lambda" not in keyvalue:
            raise ValueError("(3) Missing breakpoint wavelengths") 
        lam = list_check(keyvalue["pl_lambda"])
        if len(lam) != n+1:
            raise ValueError("(3) Need N+1 breakpoint wavelengths values")
        if any(1<0.1 for l in lam) or any(l>1 for l in lam):
            raise ValueError("(3) Breakpoint wavelengths values must be between 0.1 and 1 microns")
        increasing(lam)

        if "pl_k" not in keyvalue:
            raise ValueError("(3) Missing power indices (k)")
        Ks = list_check(keyvalue["pl_k"])
        if len(lam) != n+1:
            raise ValueError("(3) Need the same number of power indices (k) values as number of power law segmenets")

    elif spectrum in (4,5,6):
        with open(inp, 'r', encoding='utf-8') as file:
            for line in file:
                if 'spectrum' in line:
                    try:
                        next_line_spec = next(file)
                        stellar_spectrum_dat_check(next_line_spec)
                    except Exception as err:
                        print(err)

    
    #Chemical Composition
    if "optical_properties_index" not in keyvalue: 
        raise ValueError("Missing optical properties index information or heading")
    opt = find_num(keyvalue["optical_properties_index"])
    if opt not in (1, 2, 3):
        raise ValueError("Optical properties index type must be an integer up to value of 3")

    if opt == 1:
        if "comp_array" not in keyvalue:
            raise ValueError("Missing flag for tabulated chemical composition")
        comp = keyvalue["comp_array"].strip()
        try:
            values = [float(x) for x in comp.split(',')]
            if len(values) != 6:
                raise ValueError("Include six abundances for chemical composition")
            for num in values:
                if num > 1:
                    raise ValueError("Abundances must be less than 1")
        except Exception as err:
            print(err)
            
    if opt == 2:
        if "comp_array" not in keyvalue:
            raise ValueError("Missing flag for tabulated chemical composition")
        comp = keyvalue["comp_array"].strip()
        try:
            values = [float(x) for x in comp.split(',')]
            if len(values) != 6:
                raise ValueError("Include six abundances for chemical composition")
            for num in values:
                if num > 1:
                    raise ValueError("Abundances must be less than 1")
        except Exception as err:
            print(err)
            
    if opt == 3:
        with open(inp, 'r', encoding='utf-8') as file:
            for line in file:
                if 'optical_properties_index' in line:
                    try:
                        next_line = next(file)
                        chemical_comp_dat_check(next_line)
                    except Exception as err:
                        print(err)

    
    #Grain size distribution
    if "size_distribution" not in keyvalue: 
        raise ValueError("Missing size distribution information or heading")
    grain = find_num(keyvalue["size_distribution"])
    if grain not in (1, 2, 3):
        raise ValueError("Size distribution type must be an integer between 1 and 3")

    if grain == 2:
        if "mrn_expo" not in keyvalue:
            raise ValueError("(2) Missing Power Index (q)")
        q = find_num(keyvalue["mrn_expo"])
        if "mrn_a_min" not in keyvalue:
            raise ValueError("(2) Missing a(min)")
        a_min = find_num(keyvalue["mrn_a_min"])
        if "mrn_a_max" not in keyvalue:
            raise ValueError("(2) Missing a(max)")
        a_max = find_num(keyvalue["mrn_a_max"])
        if a_max<a_min:
            raise ValueError("(2) a(max) must be greater than a(min)")

    if grain == 3:
        if "kmh_expo" not in keyvalue:
            raise ValueError("(3) Missing Power Index (q)")
        q = find_num(keyvalue["kmh_expo"])
        if "kmh_a_min" not in keyvalue:
            raise ValueError("(3) Missing Lower Limit (a(min)")
        a_min = find_num(keyvalue["kmh_a_min"])
        if "kmh_a0" not in keyvalue:
            raise ValueError("(3) Missing characteristic size (a0)")
        a0 = find_num(keyvalue["kmh_a0"])


    #Dust Temperature
    if "density_type" not in keyvalue: 
        raise ValueError("Missing boundary temperature information or heading")
    temp_inner = find_num(keyvalue["temp_inner"])

    
    #Density Distribution
    if "density_type" not in keyvalue: 
        raise ValueError("Missing density distribution information or heading")
    density = find_num(keyvalue["density_type"])
    if density not in (0, 1, 2, 3, 4, 5):
        raise ValueError("Density distribution type must be an integer between 0 and 5")
    
    if density == 0:
        if "cos(angle)" not in keyvalue:
            raise ValueError("(0) Missing cos(angle) ")
        cos = find_num(keyvalue["cos(angle)"])

        if "r" not in keyvalue:
            raise ValueError("(0) Missing R")
        r = find_num(keyvalue["r"])
        if r != 0:
            if "cos(angle)" not in keyvalue:
                raise ValueError("(0) Missing cos(angle) for source on the right")
            if "spectrum" not in keyvalue:
                raise ValueError("Missing spectral type of the right hand source")
            if "n" not in keyvalue:
                raise ValueError("Missing number of black bodies for the right hand source")
            if "tbb" not in keyvalue:
                raise ValueError("Missing the temperature of black bodies for the right hand source")

    if density == 1:
        if "N_value" not in keyvalue:
            raise ValueError("(1) Missing number of shell pieces (N_value) for power law")
        n = find_num(keyvalue["N_value"])
        if "transition_radii" not in keyvalue:
            raise ValueError("Missing transition radii")
        radii = list_check(keyvalue["transition_radii"])
        if len(radii) != n:
            raise ValueError("Number of transition radii must correspond to given N")
        if "power_indices" not in keyvalue:
            raise ValueError("Missing power indices")
        indices = list_check(keyvalue["power_indices"])
        if len(indices) != n:
            raise ValueError("Number of power indices must correspond to given N")

    if density == 2:
        if "radius" not in keyvalue:
            raise ValueError("Missing radius for exponentially decreasing density distribution")
        if "sigma" not in keyvalue:
            raise ValueError("Missing sigma for exponentially decreasing density distribution")

    if density == 3:
        if "radius" not in keyvalue:
            raise ValueError("Missing radius for numerical solution for radiatively driven winds density distribution")

    if density == 4:
        if "radius" not in keyvalue:
            raise ValueError("Missing radius for analytical approximation for radiatively driven winds density distribution")

    if density == 5:
        with open(inp, 'r', encoding='utf-8') as file:
            for line in file:
                if 'density_type' in line:
                    try:
                        next_line = next(file)
                        density_dat_check(next_line)
                    except Exception as err:
                        print(err)


        
    
    #Optical Depth
    if "tau_grid" not in keyvalue: 
        raise ValueError("Missing Grid type information or heading")
    grid = find_num(keyvalue["tau_grid"])
    if int(grid) not in (1, 2, 3):
        raise ValueError("Grid type must be an integer up to value of 3")

    if grid in (1,2):
        if "lambda0" not in keyvalue:
            raise ValueError("(1,2) Missing λ0")
        lambda0 = find_num(keyvalue["lambda0"])
        if "tau_min" not in keyvalue:
            raise ValueError("(1,2) Missing τ min")
        tau_min = find_num(keyvalue["tau_min"])
        if "tau_max" not in keyvalue:
            raise ValueError("(1,2) Missing τ max")
        tau_max = find_num(keyvalue["tau_max"])
        if tau_max < tau_min:
            raise ValueError("(1,2) τ max must be greater than τ min")
        if "model_count" not in keyvalue:
            raise ValueError("(1,2) Number of models missing")
        no_models = find_num(keyvalue["model_count"])
        if no_models > 1000 or no_models < 1:
            raise ValueError("Can generate between 1 and 999 models only")

            
    if opt == 3:
        with open(inp, 'r', encoding='utf-8') as file:
            for line in file:
                if 'optical_properties_index' in line:
                    try:
                        next_line = next(file)
                        optical_dat_check(next_line)
                    except Exception as err:
                        print(err)
                        
                    
    #Numerical accuracy
    if "qacc" not in keyvalue:
        raise ValueError("Missing Numerical accuracy flag")

                    
    #Output control flag
    if "verbosity" not in keyvalue: 
        raise ValueError("Missing verbosity flag information.")
    verbose = find_num(keyvalue["verbosity"])
    if int(verbose) not in (0, 1, 2, 3):
        raise ValueError("Verbosity flag must be an integer between 0 and 3")

    if "fname_spp" not in keyvalue:
        raise ValueError("Missing fname.spp flag information.")
    fname_spp = find_num(keyvalue["fname_spp"])
    if int(fname_spp) not in (0, 1, 2, 3):
        raise ValueError("fname.spp flag must be an integer between 0 and 3")

    if "fname_sxxx" not in keyvalue:
        raise ValueError("Missing fname.s### flag information.")
    fname_s = find_num(keyvalue["fname_sxxx"])
    if int(fname_s) not in (0, 1, 2, 3):
        raise ValueError("fname.s### flag must be an integer between 0 and 3")

    if "fname_ixxx" not in keyvalue:
        raise ValueError("Missing fname.i### flag information.")
    fname_i = find_num(keyvalue["fname_ixxx"])
    if int(fname_i) not in (0, 1, 2):
        raise ValueError("fname.i### flag must be an integer between 0 and 2")
    if int(fname_i) == 1 or int(fname_i) == 2:
        if "fname_ixxx_wavelengths" not in keyvalue:
            raise ValueError("Missing number of wavelengths for fname.i### flag.")
        fname_iw = find_num(keyvalue["fname_ixxx_wavelengths"])
        if "fname_ixxx_wavelengths_array" not in keyvalue:
            raise ValueError("Missing wavelengths for fname.i### flag.")
        fname_iwa = find_list(keyvalue["fname_ixxx_wavelengths_array"])
        if len(fname_iwa) > 20:
            raise ValueError ("The number of wavelengths must be ≤ 20")
        if int(fname_iw) != len(fname_iwa):
            raise ValueError ("The number of wavelengths must match the length of the wavelength array")
        
    if "fname_rxxx" not in keyvalue:
        raise ValueError("Missing fname.r### flag information.")
    fname_r = find_num(keyvalue["fname_rxxx"])
    if int(fname_r) not in (0, 1, 2):
        raise ValueError("fname.r### flag must be an integer between 0 and 2")

    if "fname_mxxx" not in keyvalue:
        raise ValueError("Missing fname.m### flag information.")
    fname_m = find_num(keyvalue["fname_mxxx"])
    if int(fname_m) not in (0, 1, 2):
        raise ValueError("fname.m### flag must be an integer between 0 and 2")

In [5]:
check_inp('meowmeow5.inp')

Data must be in a one column format.
